In [ ]:
import pandas as pd
import geopandas as gpd

from cafo_iowa.estimate.estimate import load_and_process_facilities
from cafo_iowa.utils.utils import save_geojson_with_lists

import cafo_iowa.db.session as s
import cafo_iowa.db.models as m

In [ ]:
session = s.get_session()
engine = session.get_bind()

In [ ]:
data = gpd.read_postgis(
    f"""
    WITH
        permit_tiles AS (
            SELECT
                naip_qt_id,
                SUM(animal_units) as animal_units,
                SUM(swine_animal_units) as swine_animal_units,
                COUNT(*) as n_facilities,
                ARRAY_AGG(facilityid) as facilityid_list,
                ARRAY_AGG(id) as permitid_list,
                ARRAY_AGG(latitude) as latitude_list,
                ARRAY_AGG(longitude) as longitude_list
            FROM processed.{m.Permits.__tablename__}
            WHERE swine_animal_units > 0
            GROUP BY naip_qt_id
        ),
        naip21_qt AS (
            SELECT
                id as naip_qt_id,
                geometry,
                is_urban,
                urban_area
            FROM processed.{m.Naip21QT.__tablename__}
        ),
        labelled_tiles AS (
            SELECT
                jsonb_array_elements_text(naip_qt_ids) AS naip_qt_id,
                id as batch_nr,
                batch_name
            FROM processed.label_batches
        )
    SELECT
        n.*,
        COALESCE(p.animal_units, 0) as animal_units,
        COALESCE(p.swine_animal_units, 0) as swine_animal_units,
        COALESCE(p.n_facilities, 0) as n_facilities,
        p.facilityid_list,
        p.permitid_list,
        p.latitude_list,
        p.longitude_list,
        COALESCE(l.batch_name, 'unlabelled') as batch_name,
        l.batch_nr,
        CASE
            WHEN p.n_facilities > 0 THEN p.animal_units / p.n_facilities
            ELSE 0
        END as animal_units_per_permit,
        CASE
            WHEN p.n_facilities > 0 THEN p.swine_animal_units / p.n_facilities
            ELSE 0
        END as swine_animal_units_per_permit,
        CASE
            WHEN COALESCE(p.n_facilities, 0) = 0 THEN '0'
            WHEN p.animal_units / p.n_facilities BETWEEN 0 AND 399 THEN '1-399'
            WHEN p.animal_units / p.n_facilities BETWEEN 400 AND 499 THEN '400-499'
            WHEN p.animal_units / p.n_facilities BETWEEN 500 AND 699 THEN '500-699'
            WHEN p.animal_units / p.n_facilities BETWEEN 700 AND 899 THEN '700-899'
            WHEN p.animal_units / p.n_facilities BETWEEN 900 AND 999 THEN '900-999'
            WHEN p.animal_units / p.n_facilities BETWEEN 1000 AND 1199 THEN '1000-1199'
            WHEN p.animal_units / p.n_facilities BETWEEN 1200 AND 1499 THEN '1200-1499'
            WHEN p.animal_units / p.n_facilities BETWEEN 1500 AND 1899 THEN '1500-1899'
            WHEN p.animal_units / p.n_facilities BETWEEN 1900 AND 1999 THEN '1900-1999'
            WHEN p.animal_units / p.n_facilities BETWEEN 2000 AND 2499 THEN '2000-2499'
            WHEN p.animal_units / p.n_facilities >= 2500 THEN '2500+'
        END as animal_units_per_permit_q
    FROM naip21_qt n
    LEFT JOIN permit_tiles p USING (naip_qt_id)
    LEFT JOIN labelled_tiles l ON n.naip_qt_id = l.naip_qt_id
    """,
    s.get_engine(),
    geom_col="geometry",
)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

valid_rows = []

for idx, row in data.iterrows():
    tile_geometry = row["geometry"]
    latitudes = row["latitude_list"]
    longitudes = row["longitude_list"]

    if latitudes is None or longitudes is None:
        continue
    found_valid_permit = False

    tile_gdf = gpd.GeoDataFrame(
        geometry=[tile_geometry], crs=data.crs
    )  # Use the CRS of the data GeoDataFrame

    for lat, lon in zip(latitudes, longitudes):
        if lat is None or lon is None:
            continue
        permit_point = Point(lon, lat)

        permit_gdf = gpd.GeoDataFrame(geometry=[permit_point], crs="EPSG:4326")
        permit_gdf = permit_gdf.to_crs(tile_gdf.crs)

        distance_to_edge = tile_gdf.boundary.distance(permit_gdf).values[0]
        if distance_to_edge <= 100:
            found_valid_permit = True
            break
    if found_valid_permit:
        valid_rows.append(idx)

filtered_data = data.loc[valid_rows]

In [ ]:
filtered_data

In [ ]:
filtered_data["naip_qt_id"].tolist()

In [ ]:
import matplotlib.pyplot as plt

# Tile number
tile_num = 150

geometry = filtered_data.iloc[tile_num]["geometry"]
latitudes = filtered_data.iloc[tile_num]["latitude_list"]
longitudes = filtered_data.iloc[tile_num]["longitude_list"]
for lat, lon in zip(latitudes, longitudes):
    if lat is None or lon is None:
        continue
    permit_point = Point(lon, lat)

    permit_gdf = gpd.GeoDataFrame(geometry=[permit_point], crs="EPSG:4326")
    permit_gdf = permit_gdf.to_crs(filtered_data.crs)


distance_to_edge = geometry.boundary.distance(permit_gdf).values[0]
fig, ax = plt.subplots(figsize=(10, 10))
geo_series = gpd.GeoSeries([geometry])
geo_series.plot(ax=ax, color="lightblue", edgecolor="black", alpha=0.5)
permit_gdf.plot(ax=ax, color="red", marker="o", markersize=20)